# Maintenance Tickets Preprocessing

Cleans and engineers features from maintenance ticket and event-log data: ticket volumes,
fixing/closure status, contractor and part wait times, schedule dates, and ticket-type
features, joined with site-to-region mapping.

Stack: PySpark, Microsoft Fabric lakehouse.

Note: internal identifiers are anonymized, and outputs are cleared because the originals
ran on internal data.


In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from datetime import datetime
import pytz
from pyspark.sql.functions import col, to_date, split, to_timestamp, when, date_format, lit
from pyspark.sql.types import IntegerType, StringType, StructType, StructField, TimestampType
from pyspark.sql import functions as F

# Load Data

In [ ]:
from pyspark.sql.functions import regexp_replace
# Read the file as text to inspect the first few lines

events_spark = spark.sql("SELECT event_id, classification, created_at, audited_changes \
                          FROM Bronze_RawData.Bronze_Lakehouse.rp.Events \
                          WHERE event_type = 'MaintenanceTicket'")

# Convert 'created_at' column to timestamp
events_spark = events_spark.withColumn("created_at", col("created_at").cast("timestamp"))

# Remove backticks around the text in the 'classification' column
events_spark = events_spark.withColumn('classification', regexp_replace('classification', '`', ''))
events_spark = events_spark.withColumn('audited_changes', regexp_replace('audited_changes', '`', ''))

In [ ]:
from pyspark.sql.functions import col, to_date, split, to_timestamp

# Read the delata table from lakehouse
original_tickets_spark = spark.sql("SELECT * FROM Silver_Lakehouse.rp.Custom_maintenance_tickets")

# Keep only the useful columns
useful_cols = ['id', 'created_at', 'request_type', 'priority',
               'completion_date', 'updated_at', 'ticket_description', 'subject', 'category',
               'suite_location', 'subcategory', 'internal_notes', 'status', 'unit_id', 
               'property_id', 'ticket_number', 'tenant_note', 'classification', 
               'created_by_type', 'unit_number', 'time_zone', 'assigned_to_name','sap_building_id']
tickets_spark = original_tickets_spark.select(*useful_cols)

# Extract the 'site' from 'external_id'
tickets_spark = tickets_spark.withColumn('site', split(col('sap_building_id'), '/').getItem(1))
tickets_spark = tickets_spark.withColumn('Building_Number', split(col('sap_building_id'), '/').getItem(2))

# Convert 'site' to integer and then to string
tickets_spark = tickets_spark.withColumn('site', col('site').cast(IntegerType()).cast(StringType()))

# Convert datetime columns while retaining time elements
tickets_spark = tickets_spark.withColumn('created_at', to_timestamp(col('created_at')))
tickets_spark = tickets_spark.withColumn('completion_date', to_timestamp(col('completion_date')))

# drop duplicates based on id
tickets_spark = tickets_spark.dropDuplicates(["id"])

### Events logs data

In [ ]:
# from pyspark.sql.functions import col

# # Filter the events DataFrame for the specified event_id and sort by created_at in descending order
# filtered_events = events_spark.filter(col('event_id') == 734430).orderBy(col('created_at').desc())

# # Collect the filtered DataFrame to the driver
# filtered_events_collected = filtered_events.collect()

# # Iterate through the collected rows and print the required fields
# for row in filtered_events_collected:
#     print(row['created_at'])
#     print(row['classification'])
#     print(row['audited_changes'])
#     print('------------------')

## Hierarchy Data

In [ ]:
# Reading data for mapping building to region
region_spark = spark.sql("SELECT * FROM Silver_Lakehouse.dbo.DIM_RE_HIERARCHY")

# Iterate over all columns and replace blank values with null values
for column in region_spark.columns:
    region_spark = region_spark.withColumn(column, when(col(column) == '', None).otherwise(col(column)))

# Exclude commercial and landlease
region_spark = region_spark.filter((col('Building_Class')!= "COMMERCIAL") & (col('Building_Class')!= "LANDLEASE"))

# only usefull columns
cols = ['Building', 'Building_Number', 'Site_Number', 'Site_Number_Name', 'Acquisition_Date', 'Country',
        'Region',
        'City', 
        'House_Number',
        'Street_Name',
        'Postal_Code',
        'Building_Name',
        'Building_Class',
        'Building_Type',
        'Company',
        'Company_Code',
        'Profit_Centre_Name',
        'Profit_Centre',
        'Valid_from',
        'Valid_to',
        'Locations',
        'Suite_Count',
        'BPC_CC',
        'Construction_Category']

region_spark = region_spark.select(*cols)

region_spark = region_spark.withColumnRenamed('Site_Number', 'site')

# Data Exploration and Cleaning

### tickets

In [ ]:
from pyspark.sql.functions import col, when, to_date, date_format, lit
from pyspark.sql.types import IntegerType, StringType, StructType, StructField, TimestampType
from pyspark.sql import functions as F

# Drop rows with null values in 'site' column
tickets_spark = tickets_spark.dropna(subset=['site'])

# Convert numeric data columns to numeric
tickets_spark = tickets_spark.withColumn('id', col('id').cast(IntegerType()))

# Remove seconds and microseconds from datetime columns
tickets_spark = tickets_spark.withColumn('created_at', date_format(col('created_at'), 'yyyy-MM-dd HH:mm:00').cast(TimestampType()))
tickets_spark = tickets_spark.withColumn('completion_date', date_format(col('completion_date'), 'yyyy-MM-dd HH:mm:00').cast(TimestampType()))

# Convert 'priority' to category with specific order
priority_order = ['Normal', 'High', 'Emergency']
tickets_spark = tickets_spark.withColumn('priority', F.when(col('priority').isin(priority_order), col('priority')).otherwise(lit(None)))
tickets_spark = tickets_spark.withColumn('priority', col('priority').cast(StringType()))

# Convert specified columns to category
category_cols = ['status', 'request_type', 'category', 'classification', 'created_by_type']
for col_name in category_cols:
    tickets_spark = tickets_spark.withColumn(col_name, col(col_name).cast(StringType()))


In [ ]:
## join ticket and region data on site
from pyspark.sql.functions import regexp_extract, split

# Merge ticket and region data on site
tickets_spark = tickets_spark.join(region_spark, on=['site', 'Building_Number'], how='left')

# Extract the part after the slash in the Region column as region name
tickets_spark = tickets_spark.withColumn('Region', split(col('Region'), '/').getItem(1))

# Drop rows with null values in 'site' and 'Region' columns
tickets_spark = tickets_spark.dropna(subset=['site', 'Region'])


In [ ]:
# # Filter data

# from pyspark.sql.functions import col, current_date, add_months

# # Filter for only tickets created in the last 12 months dynamically
# tickets_spark = tickets_spark.filter(col('created_at') >= add_months(current_date(), -12))

# # If you want to filter from a specific static date ('2023-08-01')
# tickets_spark = tickets_spark.filter(col('created_at') >= '2023-08-01')

# # Display the filtered DataFrame
# display(tickets_spark)

In [ ]:
# # Data Info

# from pyspark.sql import functions as F
# from pyspark.sql import SparkSession
# from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# # Define schema for summary DataFrame
# info_schema = StructType([
#     StructField("Column", StringType(), False),
#     StructField("DataType", StringType(), False),
#     StructField("NonNullCount", IntegerType(), False),
#     StructField("UniqueCount", IntegerType(), False),
#     StructField("NullCount", IntegerType(), False)
# ])

# # Initialize an empty list to collect information
# info_data = []

# # Get data types for each column
# data_types = [(col_name, col_type) for col_name, col_type in tickets_spark.dtypes]

# # Use a single aggregation to calculate NonNullCount, UniqueCount, and NullCount in one pass
# agg_exprs = []
# for col_name, col_type in data_types:
#     agg_exprs.append(
#         F.count(F.when(F.col(col_name).isNotNull(), col_name)).alias(f"{col_name}_non_null_count")
#     )
#     agg_exprs.append(
#         F.countDistinct(col_name).alias(f"{col_name}_unique_count")
#     )
#     agg_exprs.append(
#         F.count(F.when(F.col(col_name).isNull(), col_name)).alias(f"{col_name}_null_count")
#     )

# # Perform aggregation in one go
# agg_result = tickets_spark.agg(*agg_exprs).collect()[0]

# # Parse aggregated results to create the info_data list
# for col_name, col_type in data_types:
#     non_null_count = agg_result[f"{col_name}_non_null_count"]
#     unique_count = agg_result[f"{col_name}_unique_count"]
#     null_count = agg_result[f"{col_name}_null_count"]
#     info_data.append((col_name, col_type, non_null_count, unique_count, null_count))

# # Create DataFrame to display
# info_df = spark.createDataFrame(data=info_data, schema=info_schema)

## Feature Engineering

### Tickets volume feature

In [ ]:
# Number of tickets per day
from pyspark.sql.functions import col, count, to_date
from pyspark.sql.window import Window

# Define a window specification partitioned by the date part of 'created_at'
window_spec = Window.partitionBy(to_date(col('created_at')))

# Calculate the number of tickets per day
tickets_spark = tickets_spark.withColumn('tickets_per_day', count('created_at').over(window_spec))

### Event Logs Features

In [ ]:
## Number of Fixing, fixed, and closed status for each ticket
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import ast

# Keep only the events with a classification of 'ticket_status_change'
events_status_change = events_spark.filter(F.col("classification") == "ticket_status_change")
events_status_change = events_status_change.orderBy(['event_id', 'created_at'])

# Sort values by 'event_id' and 'created_at'
window_spec = Window.partitionBy("event_id")

# Calculate the number of rows that have status "Fixing", "Fixed", and "Closed" for each 'event_id'
events_status_change = events_status_change.withColumn("fixing_count", F.sum(F.col("audited_changes").contains('"status":"Fixing"').cast("int")).over(window_spec))
events_status_change = events_status_change.withColumn("fixed_count", F.sum(F.col("audited_changes").contains('"status":"Fixed"').cast("int")).over(window_spec))
events_status_change = events_status_change.withColumn("closed_count", F.sum(F.col("audited_changes").contains('"status":"Closed"').cast("int")).over(window_spec))

# Extract the date of the first "Contractor Needed" status for each 'event_id'
contractor_needed_dates = events_status_change \
    .filter(F.col("audited_changes").contains('"status":"Contractor Needed"')) \
    .groupBy("event_id") \
    .agg(F.min("created_at").alias("need_contractor_date")) \
    .withColumn("need_contractor_date", F.to_utc_timestamp("need_contractor_date", "UTC"))

# Join back to add 'need_contractor_date' to the main DataFrame
events_status_change = events_status_change.join(contractor_needed_dates, "event_id", "left")

# Extract the date of the first "Parts Needed" status for each 'event_id'
part_needed_dates = events_status_change \
    .filter(F.col("audited_changes").contains('"status":"Parts Needed"')) \
    .groupBy("event_id") \
    .agg(F.min("created_at").alias("parts_needed_date")) \
    .withColumn("parts_needed_date", F.to_utc_timestamp("parts_needed_date", "UTC"))

# Join back to add 'parts_needed_date' to the main DataFrame
events_status_change = events_status_change.join(part_needed_dates, "event_id", "left")

# Create the sequence of status changes for each 'event_id'
# Define a UDF to extract 'status' from the JSON string in 'audited_changes'
@F.udf
def extract_status(dict_str):
    try:
        dict_obj = ast.literal_eval(dict_str)
        return dict_obj.get('status', None) if isinstance(dict_obj, dict) else None
    except:
        return None

# Apply the UDF to extract the status
events_status_change = events_status_change.withColumn("status", extract_status(F.col("audited_changes")))

# Aggregate the 'status' values into a list for each 'event_id' to create a sequence of status changes
status_change_sequence = events_status_change.groupBy("event_id").agg(F.collect_list("status").alias("status_sequence"))

# Merge the status sequence back to the main DataFrame
events_status_change = events_status_change.join(status_change_sequence, "event_id", "left")

In [ ]:
## Extracting more detailed time information regarding the tickets that need contractor
# Extracting the time of the first status change to 'Scheduled' status after 'Contractor Needed' (this is not the scheduled time)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Filter rows where status is either 'Contractor Needed' or 'Scheduled'
filtered_events = events_status_change.filter(
    F.col('status').isin(['Contractor Needed', 'Scheduled'])
)

# Sort the DataFrame by event_id and created_at
window_spec = Window.partitionBy('event_id').orderBy('created_at')

# Create a column to identify rows that are 'Scheduled' and come after 'Contractor Needed'
filtered_events = filtered_events.withColumn(
    'prev_status',
    F.lag('status').over(window_spec)
)

# Create a mask to identify rows that are 'Scheduled' and come after 'Contractor Needed'
mask = (F.col('status') == 'Scheduled') & (F.col('prev_status') == 'Contractor Needed')

# Use the mask to create a DataFrame with the first 'Scheduled' status after 'Contractor Needed'
scheduled_dates = filtered_events.filter(mask).groupBy('event_id').agg(
    F.min('created_at').alias('contractor_start_date')
).withColumn('contractor_start_date', F.to_utc_timestamp('contractor_start_date', 'UTC'))

# Merge the scheduled_dates DataFrame back to the original DataFrame
events_status_change = events_status_change.join(scheduled_dates, on='event_id', how='left')

In [ ]:
## Extracting more detailed time information regarding the tickets that need part
# Extracting the time of the first status change to 'Scheduled' status after 'Parts Needed' (this is not the scheduled time)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Filter rows where status is either 'Parts Needed' or 'Scheduled'
filtered_events = events_status_change.filter(
    F.col('status').isin(['Parts Needed', 'Scheduled'])
)

# Sort the DataFrame by event_id and created_at
window_spec = Window.partitionBy('event_id').orderBy('created_at')

# Create a column to identify rows that are 'Scheduled' and come after 'Contractor Needed'
filtered_events = filtered_events.withColumn(
    'prev_status',
    F.lag('status').over(window_spec)
)

# Create a mask to identify rows that are 'Scheduled' and come after 'Contractor Needed'
mask = (F.col('status') == 'Scheduled') & (F.col('prev_status') == 'Parts Needed')

# Use the mask to create a DataFrame with the first 'Scheduled' status after 'Contractor Needed'
scheduled_dates = filtered_events.filter(mask).groupBy('event_id').agg(
    F.min('created_at').alias('part_start_date')
).withColumn('part_start_date', F.to_utc_timestamp('part_start_date', 'UTC'))

# Merge the scheduled_dates DataFrame back to the original DataFrame
events_status_change = events_status_change.join(scheduled_dates, on='event_id', how='left')

In [ ]:
## Extracting the schedule time (response date) and schedule date (the time set for maintenance)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import json
from pyspark.sql.types import StringType

# Filter events for 'ticket_scheduled'
events_scheduled = events_spark.filter(F.col("classification") == "ticket_scheduled")

# Count the occurrences of each event_id
event_counts = events_scheduled.groupBy("event_id").count().withColumnRenamed("count", "scheduled_count")

# Join the count back to the original DataFrame
events_scheduled = events_scheduled.join(event_counts, on="event_id", how="left")

# Function to extract 'scheduled_date' and 'scheduled_time' from 'audited_changes'
def extract_scheduled_date(dict_str):
    dict_obj = json.loads(dict_str)
    return dict_obj.get('scheduled_date', None) if isinstance(dict_obj, dict) else None

def extract_scheduled_time(dict_str):
    dict_obj = json.loads(dict_str)
    return dict_obj.get('scheduled_time', None) if isinstance(dict_obj, dict) else None

# Register the functions as UDFs
extract_scheduled_date_udf = F.udf(extract_scheduled_date, StringType())
extract_scheduled_time_udf = F.udf(extract_scheduled_time, StringType())

# Apply the UDFs to extract 'scheduled_date' and 'scheduled_time'
events_scheduled = events_scheduled.withColumn("scheduled_date", extract_scheduled_date_udf(F.col("audited_changes")))
events_scheduled = events_scheduled.withColumn("scheduled_time", extract_scheduled_time_udf(F.col("audited_changes")))

# Convert scheduled_time to 24-hour format
events_scheduled = events_scheduled.withColumn(
    "scheduled_time_24",
    F.date_format(F.to_timestamp(F.col("scheduled_time"), "h:mm a"), "HH:mm:ss")
)

# Create 'scheduled_datetime' column
events_scheduled = events_scheduled.withColumn(
    "scheduled_datetime",
    F.when(
        F.col("scheduled_date").isNotNull() & F.col("scheduled_time").isNotNull(),
        F.to_timestamp(F.concat(F.col("scheduled_date"), F.lit(" "), F.col("scheduled_time_24")))
    ).when(
        F.col("scheduled_date").isNotNull() & F.col("scheduled_time").isNull(),
        F.to_timestamp(F.concat(F.col("scheduled_date"), F.lit(" 09:00:00")))
    ).otherwise(F.lit(None))
)

# Sort the DataFrame by 'scheduled_datetime'
window_spec = Window.partitionBy("event_id").orderBy("scheduled_datetime")

# Get the first response and schedule date for each event
scheduled_min_dates = events_scheduled.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .select("event_id", F.col("created_at").alias("first_response_at"), F.col("scheduled_datetime").alias("first_scheduled_at"), "scheduled_count")

# Get the last response and schedule date for each event
window_spec_desc = Window.partitionBy("event_id").orderBy(F.col("scheduled_datetime").desc())
scheduled_max_dates = events_scheduled.withColumn("row_num", F.row_number().over(window_spec_desc)) \
    .filter(F.col("row_num") == 1) \
    .select("event_id", F.col("created_at").alias("last_response_at"), F.col("scheduled_datetime").alias("last_scheduled_at"))

# Merge the new information back to the original DataFrame
events_status_change = events_status_change.join(scheduled_min_dates, on="event_id", how="left")
events_status_change = events_status_change.join(scheduled_max_dates, on="event_id", how="left")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql import Window

# Define the UDF to count reopens
def count_reopens(status_list):
    reopen_count = 0
    for i in range(len(status_list) - 1):
        if status_list[i] == "Closed" and i < len(status_list):
            reopen_count += 1
        elif status_list[i] == "Fixed" and status_list[i+1] != "Closed":
            reopen_count += 1
    return reopen_count

# Register the UDF
count_reopens_udf = F.udf(count_reopens, IntegerType())

# Apply the UDF to the 'status_sequence' column
events_status_change = events_status_change.withColumn("reopen_count", count_reopens_udf(F.col("status_sequence")))

In [ ]:
from pyspark.sql import functions as F

# Select the required features and drop duplicates based on 'event_id'
features = ['event_id', 'fixing_count', 'fixed_count', 'closed_count', 'scheduled_count',
            'reopen_count', 'status_sequence', 'first_response_at', 'first_scheduled_at', 'last_response_at', 'last_scheduled_at',
            'need_contractor_date', 'contractor_start_date', 'parts_needed_date', 'part_start_date']

events_features = events_status_change.select(features).dropDuplicates(['event_id'])

# Merge the ticket data with the new features from the events data
ticket = tickets_spark.join(events_features, tickets_spark.id == events_features.event_id, how='left').drop('event_id')

In [ ]:
## creating some boolean feature
from pyspark.sql import functions as F

# Whether needs part or not
ticket = ticket.withColumn(
    'needs_part',
    F.expr("array_contains(status_sequence, 'Parts Needed')")
)

# Whether needs contractor or not
ticket = ticket.withColumn(
    'needs_contractor',
    F.expr("array_contains(status_sequence, 'Contractor Needed')")
)

# Whether parts need review or not
ticket = ticket.withColumn(
    'part_needs_review',
    F.expr("array_contains(status_sequence, 'Parts Needed Review')")
)

# Whether contractor needs review or not
ticket = ticket.withColumn(
    'contractor_needs_review',
    F.expr("array_contains(status_sequence, 'Contractor Needed Review')")
)

### Date and Time Features

In [ ]:
from pyspark.sql import functions as F

# Calculate resolution time as the difference between created_at and completion_date (if available) or updated_at (if completion_date is not available)
ticket = ticket.withColumn(
    'resolution_time',
    F.when(F.col('completion_date').isNotNull(), 
           (F.col('completion_date').cast('long') - F.col('created_at').cast('long')) / 86400)
     .otherwise(F.lit(None))
)

# Calculate response time as the time interval between created_at and first_response_at
ticket = ticket.withColumn(
    'response_time',
    F.when(F.col('first_scheduled_at').isNotNull(), 
           (F.col('first_response_at').cast('long') - F.col('created_at').cast('long')) / 86400)
     .otherwise(F.lit(0))
)

# Calculate maintenance time as the time interval between scheduled date and completion date
ticket = ticket.withColumn(
    'maintenance_time',
    (F.col('resolution_time') - F.col('response_time'))
)

# Calculate the time interval between the first response and the time of asking for contractor
ticket = ticket.withColumn(
    'contractor_need_time',
    (F.col('need_contractor_date').cast('long') - F.col('created_at').cast('long')) / 86400
)

ticket = ticket.withColumn(
    'contractor_finding_time',
    (F.col('contractor_start_date').cast('long') - F.col('need_contractor_date').cast('long')) / 86400
)

# Calculate the time interval between the first response and the time of asking for parts
ticket = ticket.withColumn(
    'part_need_time',
    (F.col('parts_needed_date').cast('long') - F.col('created_at').cast('long')) / 86400
)

ticket = ticket.withColumn(
    'part_finding_time',
    (F.col('part_start_date').cast('long') - F.col('parts_needed_date').cast('long')) / 86400
)

# Ensure all time columns are non-negative
ticket = ticket.withColumn('resolution_time', F.when(F.col('resolution_time') < 0, 0).otherwise(F.col('resolution_time')))
ticket = ticket.withColumn('response_time', F.when(F.col('response_time') < 0, 0).otherwise(F.col('response_time')))
ticket = ticket.withColumn('maintenance_time', F.when(F.col('maintenance_time') < 0, 0).otherwise(F.col('maintenance_time')))
ticket = ticket.withColumn('contractor_need_time', F.when(F.col('contractor_need_time') < 0, 0).otherwise(F.col('contractor_need_time')))
ticket = ticket.withColumn('contractor_finding_time', F.when(F.col('contractor_finding_time') < 0, 0).otherwise(F.col('contractor_finding_time')))
ticket = ticket.withColumn('part_need_time', F.when(F.col('part_need_time') < 0, 0).otherwise(F.col('part_need_time')))
ticket = ticket.withColumn('part_finding_time', F.when(F.col('part_finding_time') < 0, 0).otherwise(F.col('part_finding_time')))

### Features related to type of ticket

In [ ]:
# Drop rows with null values in the 'category' column
ticket = ticket.dropna(subset=['category'])

# Replace backticks in the 'category' column
ticket = ticket.withColumn('category', F.regexp_replace('category', '`', ''))

In [ ]:
# Clean category column
ticket = ticket.withColumn(
    'category',
    F.when(
        F.col('category').startswith('Rental Home'),
        F.split(F.col('category'), ' - ').getItem(1)
    ).otherwise(F.col('category'))
)

In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Define the rename dictionaries
rename_dict_cat = {
    'Locks/Keys (Common Areas)': 'Locks/Keys - Common Areas',
    'Locks/Keys (Units)': 'Locks/Keys - Units',
    'Security Systems (Equipment)': 'Security Systems - Equipment',
    'Equipment (Operational Check)': 'Equipment Operational Check',
    'Sewer (Backup Exterior)': 'Sewer Backup Exterior',
    'Sewer Backup Interior': 'Sewer Backup Interior',
    'Road (Pothole)': 'Roads (Pothole)',
    'Road (Other)': 'Roads (Other)',
    'Paint/Plaster (Touch-Ups)': 'Paint/Plaster - Touch-ups',
    'Paint/Plaster (Large)': 'Paint/Plaster - Large',
    'Electrical(Light Switches/Plugs/Covers)': 'Electrical (Light Switches/Plugs/Covers)',
    'Heating/Tracing Cables': 'Heating (Tracing Cables)',
    'Party Room Appliance': 'Appliances (Party Room)',
    'Appliance': 'Appliances',
    'Appliance (Dishwasher)': 'Appliances (Dishwasher)',
    'Appliance (Fridge)': 'Appliances (Fridge)',
    'Appliance (Washer/Dryer)': 'Appliances (Washer/Dryer)',
    'Appliance (Stove)': 'Appliances (Stove)',
    'Appliance (Hood Range)': 'Appliances (Hood Range)'
}

rename_dict_super = {
    'General': 'General Cleaning and Garbage Removal',
    'Fans': 'Fans Check/Clean',
    'Health': 'Health and Safety',
    'Security': 'Security Systems',
    'Grass': 'Grass/Shrubbery',
    'Road': 'Roads',
    'Doors/Windows': 'Doors and Windows',
    'Heating': 'Heating/Cooling',
    'Doors': 'Doors and Windows',
    'Admin': 'Admin (Add Phone # to Intercom)',
    'Health & Safety': 'Health and Safety',
    'Sewer Backup Exterior': 'Sewer',
    'Sewer Backup Interior': 'Sewer',
    'Window/Patio Door': 'Window/Patio Door/Screen'
}

# Replace categories based on rename_dict_cat
for old_cat, new_cat in rename_dict_cat.items():
    ticket = ticket.withColumn('category', F.when(F.col('category') == old_cat, new_cat).otherwise(F.col('category')))

# Define a UDF to determine the super category based on the category column
def get_super_category(category):
    # Split by "(" or "-" and take the first part
    s = re.split(r'\(|-', category)[0]
    # Strip to remove any trailing spaces
    return s.strip()

get_super_category_udf = F.udf(get_super_category, StringType())

# Apply the UDF to the category column and create a new super_category column
ticket = ticket.withColumn('super_category', get_super_category_udf(F.col('category')))

# # Replace super categories based on rename_dict_super
# for old_super, new_super in rename_dict_super.items():
#     ticket = ticket.withColumn('super_category', F.when(F.col('super_category') == old_super, new_super).otherwise(F.col('super_category')))

# Updating the tickets table in lakehouse

In [ ]:
# Create the schema if it does not exist
spark.sql("CREATE SCHEMA IF NOT EXISTS rp")

# Save the DataFrame as a table in the specified schema
# table_name = "rp.maintenance_tickets_processed"
ticket.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').save("abfss://Silver_TransformedData@onelake.dfs.fabric.microsoft.com/Silver_Lakehouse.Lakehouse/Tables/rp/maintenance_tickets_processed")